In [14]:
# Importar as bibliotecas

import json
from pathlib import Path
import os
import urllib.request
import sys

In [15]:
# Resolver os 'imports' do projeto 

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

from utils.classes.ipea_link import IpeaLink
from utils.classes.pandas_dataframe import PandasDataframe as pd_dataframe

In [16]:
# Traz as opções de configuração do JSON

json_file = os.path.abspath('../../../../options.json')

with open(json_file, 'r', encoding='utf-8') as j_file:
    json_data = json.load(j_file)

In [17]:
# Declaração dos caminhos

dir_data = os.path.join(os.path.abspath('../../../../'), 'data')

In [18]:
# Criação das pastas 'data'

if not os.path.exists(dir_data):
    os.mkdir(dir_data)

dir_metrics = os.path.join(dir_data, 'metrics')

if not os.path.exists(dir_metrics):
    os.mkdir(dir_metrics)

dir_cdi = os.path.join(dir_metrics, json_data['DIR']['METRICS']['CDI'])

if not os.path.exists(dir_cdi):
    os.mkdir(dir_cdi)

In [ ]:
# Compilar os dados do CDI

url = IpeaLink("('SGS366_CDI366')", "/Valores")
url = url.create_link()

with urllib.request.urlopen(url) as response:
    body = response.read().decode('utf-8')
    full_data = json.loads(body)

body = full_data['value']

cdi_date_list = []
cdi_tax_list = []

for row in body:
   
    date = str(row['VALDATA'])[:10]
    cdi_date_list.append(date)

    CDI_tax = str(row['VALVALOR'])
    cdi_tax_list.append(CDI_tax)

cdi_dictionary = { "DATE": cdi_date_list, "CDI_TAX": cdi_tax_list }

In [21]:
# Criar uma planilha com os dados diários do CDI

archive_name = 'cdi_data.csv'
ARCHIVE_PATH = os.path.join(dir_cdi, archive_name)

df_cdi = pd_dataframe(None, None, dict=cdi_dictionary)
df_cdi.dict_to_df()
df_cdi.df_to_csv(ARCHIVE_PATH)